# Extract the two approved Lanna dictionaries
This notebook creates a resumable **review queue**. It never adds OCR output to training automatically. Every accepted row must be compared with the page image and marked `approved`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/translation_model'
!pip -q install 'transformers>=4.49.0' accelerate bitsandbytes qwen-vl-utils pypdfium2 pillow requests gdown


In [ ]:
import io, json, os, re, requests, torch
from pathlib import Path
from PIL import Image
import pypdfium2 as pdfium
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

RAW_DIR = Path(PROJECT_DIR) / 'data/raw/external'
SOURCE_DIR = Path(PROJECT_DIR) / 'data/sources'
RAW_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
QUEUE_PATH = RAW_DIR / 'review_queue.jsonl'
DONE_PATH = RAW_DIR / 'completed_segments.json'
PDF_PATH = SOURCE_DIR / 'lanna_dictionary_drive_672.pdf'
if not PDF_PATH.exists():
    !gdown --id 1vU2y-5V0uRyY0FKQ74m4h2fJVScwwHJL -O "{PDF_PATH}"
print(PDF_PATH, PDF_PATH.stat().st_size)


## OCR model
A compact vision-language model is used only to create candidates. Tai Tham is a low-resource script, so valid Unicode alone is not proof that the spelling is correct.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map='auto', quantization_config=quant, torch_dtype=torch.float16
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
PROMPT = '''You are transcribing one column of a printed Lanna dictionary.
Return only a JSON array. Each object must have exactly: thai, lanna, pronunciation, meaning.
- lanna must be an exact transcription in Unicode Tai Tham U+1A20-U+1AAF.
- Never transliterate, normalize spelling, invent a missing glyph, or output legacy Thai-font codes.
- thai is the Thai headword or Thai equivalent.
- pronunciation is the pronunciation printed on the page.
- meaning is the Thai definition printed on the page.
- If any field cannot be read exactly, use null for that field.
- Ignore headings, page numbers, and English definitions.'''


In [ ]:
def parse_json_array(text):
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip(), flags=re.I | re.S)
    match = re.search(r'\[.*\]', text, flags=re.S)
    if not match:
        return []
    try:
        value = json.loads(match.group(0))
        return value if isinstance(value, list) else []
    except json.JSONDecodeError:
        return []

def infer(image):
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image}, {'type': 'text', 'text': PROMPT}
    ]}]
    chat = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[chat], images=image_inputs, videos=video_inputs, padding=True, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        ids = model.generate(**inputs, max_new_tokens=2048, do_sample=False)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, ids)]
    return parse_json_array(processor.batch_decode(trimmed, skip_special_tokens=True)[0])

def split_columns(image):
    width, height = image.size
    overlap = max(40, width // 30)
    middle = width // 2
    return [('left', image.crop((0, 0, middle + overlap, height))), ('right', image.crop((middle - overlap, 0, width, height)))]

def tai_tham_count(value):
    return sum(0x1A20 <= ord(ch) <= 0x1AAF for ch in str(value or ''))

def append_records(records):
    with QUEUE_PATH.open('a', encoding='utf-8') as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + '\n')


## Choose a source and page range
Run smaller ranges first (for example 20 pages), review them, and then continue. The progress file prevents completed page-columns from running twice.

In [ ]:
SOURCE = 'drive'      # 'drive' or 'anyflip'
START_PAGE = 1
END_PAGE = 20         # drive: max 672, anyflip: max 421

completed = set(json.loads(DONE_PATH.read_text(encoding='utf-8'))) if DONE_PATH.exists() else set()
pdf = pdfium.PdfDocument(str(PDF_PATH)) if SOURCE == 'drive' else None
for page_number in range(START_PAGE, END_PAGE + 1):
    if SOURCE == 'drive':
        page = pdf[page_number - 1]
        image = page.render(scale=2.2).to_pil().convert('RGB')
        source_id = 'lanna_dictionary_drive_672'
    else:
        url = f'https://online.anyflip.com/bvgql/goim/files/mobile/{page_number}.webp'
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        image = Image.open(io.BytesIO(response.content)).convert('RGB')
        source_id = 'wanchai_lanna_thai_anyflip_421'
    for column, crop in split_columns(image):
        segment_id = f'{source_id}:{page_number}:{column}'
        if segment_id in completed:
            continue
        candidates = infer(crop)
        rows = []
        for candidate in candidates:
            if not isinstance(candidate, dict):
                continue
            rows.append({
                'thai': candidate.get('thai'),
                'lanna': candidate.get('lanna'),
                'pronunciation': candidate.get('pronunciation'),
                'meaning': candidate.get('meaning'),
                'source_id': source_id,
                'source_page': page_number,
                'source_column': column,
                'machine_tai_tham_characters': tai_tham_count(candidate.get('lanna')),
                'review_status': 'pending'
            })
        append_records(rows)
        completed.add(segment_id)
        DONE_PATH.write_text(json.dumps(sorted(completed), ensure_ascii=False, indent=2), encoding='utf-8')
        print(segment_id, len(rows))


## After human review
Open `review_queue.jsonl`, compare every row with the source page, correct all fields, and change only verified rows to `review_status: approved`. Then validate and rebuild the training data.

In [ ]:
!python "{PROJECT_DIR}/scripts/validate_review_queue.py" --input "{RAW_DIR}/review_queue.jsonl" --accepted "{RAW_DIR}/approved_records.jsonl" --rejected "{RAW_DIR}/rejected_records.jsonl" --report "{PROJECT_DIR}/reports/external_validation_report.json"
!python "{PROJECT_DIR}/scripts/prepare_dataset.py" --input "{PROJECT_DIR}/data/raw/lanna_dict.json" --verified-overrides "{PROJECT_DIR}/data/raw/verified_overrides.json" --external-input "{RAW_DIR}/approved_records.jsonl" --output-dir "{PROJECT_DIR}/data/processed"
print('Training splits rebuilt. Open colab_train_byt5.ipynb to fine-tune the translation model.')
